# Bài 19: Phân cụm — k-Means, GMM và DBSCAN (Clustering — k-Means, GMMs, and DBSCAN)

Mọi bài trước trong phần này đều giả định có một *mục tiêu* nào đó — một ngưỡng, một khuôn mẫu, một phép biến đổi đã biết. **Phân cụm (clustering)** đặt ra một loại câu hỏi khác, hoàn toàn không có mục tiêu: cho một đống điểm (hoặc điểm ảnh), hãy nhóm chúng thành một số ít loại hợp lý, chỉ dựa vào mức độ giống nhau giữa chúng. Đây là điểm khởi đầu kinh điển của **học không giám sát (unsupervised learning)** — không có nhãn, chỉ có một hàm khoảng cách và một giả định về việc "một cụm" nên trông như thế nào. Bài này xây dựng từ đầu ba thuật toán phân cụm, mỗi thuật toán dựa trên một giả định khác nhau về hình dạng cụm, và chỉ ra cụ thể chỗ mỗi thuật toán thành công và thất bại: **k-means** (các cụm tròn, kích thước tương tự nhau), **mô hình hỗn hợp Gauss (Gaussian mixture models)** (các cụm elip, gán mềm), và **DBSCAN** (hình dạng tùy ý, được xác định bởi mật độ thay vì một tâm).

In [ ]:
import numpy as np
import cv2
import matplotlib.pyplot as plt

## k-means trên dữ liệu màu thật (k-means on real color data)
Bài 18 đã xây dựng một khoảng cách màu có ý nghĩa về cảm nhận (khoảng cách Euclid L\*a\*b\*). Phân cụm là bước tự nhiên tiếp theo: cho một đống màu điểm ảnh và hoàn toàn không có nhãn, hãy tự động nhóm chúng thành các vùng. Hãy tạo một ảnh màu tổng hợp gồm 4 vùng (có nhiễu, để không thể tách đơn giản bằng cách khớp màu chính xác), chuyển sang L*a*b*, và phân cụm các màu điểm ảnh thô.

In [ ]:
SIZE = 48

def make_region_image(size=SIZE, noise_std=8, seed=3):
    rng = np.random.default_rng(seed)
    img = np.zeros((size, size, 3), dtype=np.uint8)
    region_map = np.zeros((size, size), dtype=np.int64)
    colors_bgr = [(180, 40, 30), (40, 160, 60), (30, 80, 190), (200, 200, 60)]
    centers = [(12, 12), (12, 36), (36, 12), (36, 36)]
    yy, xx = np.mgrid[0:size, 0:size]
    for k, ((cy, cx), col) in enumerate(zip(centers, colors_bgr)):
        mask = (xx - cx) ** 2 + (yy - cy) ** 2 <= 20 ** 2
        img[mask] = col
        region_map[mask] = k + 1
    noise = rng.normal(0, noise_std, img.shape)
    img = np.clip(img.astype(np.float64) + noise, 0, 255).astype(np.uint8)
    return img, region_map

img, region_map = make_region_image()
lab = cv2.cvtColor(img, cv2.COLOR_BGR2LAB).astype(np.float64)
X_pixels = lab.reshape(-1, 3)

def kmeans(X, k, n_iter=50, seed=0):
    rng = np.random.default_rng(seed)
    centers = X[rng.choice(len(X), k, replace=False)].copy()
    for _ in range(n_iter):
        dists = ((X[:, None, :] - centers[None, :, :]) ** 2).sum(-1)
        labels = dists.argmin(1)
        new_centers = np.array([X[labels == j].mean(0) if (labels == j).any() else centers[j]
                                 for j in range(k)])
        if np.allclose(new_centers, centers):
            break
        centers = new_centers
    return labels, centers

def purity(labels, true_flat):
    correct = 0
    for u in np.unique(labels):
        mask = labels == u
        majority = np.bincount(true_flat[mask]).argmax()
        correct += (true_flat[mask] == majority).sum()
    return correct / len(true_flat)

km_labels, _ = kmeans(X_pixels, k=4, seed=3)
km_seg = km_labels.reshape(SIZE, SIZE)
print(f'k-means segmentation purity vs. true regions: {purity(km_labels, region_map.ravel()):.3f}')

fig, axes = plt.subplots(1, 3, figsize=(9, 3.2))
axes[0].imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB)); axes[0].set_title('input (noisy)', fontsize=9); axes[0].axis('off')
axes[1].imshow(region_map, cmap='tab10'); axes[1].set_title('true regions', fontsize=9); axes[1].axis('off')
axes[2].imshow(km_seg, cmap='tab10'); axes[2].set_title('k-means segmentation', fontsize=9); axes[2].axis('off')
plt.show()

Ảnh động này cho thấy thuật toán Lloyd chạy trên một đám mây điểm 2D nhỏ (cố tình khởi tạo tâm kém, để cần vài vòng lặp): mỗi vòng luân phiên một bước **gán (assign)**, trong đó mọi điểm đổi màu theo tâm hiện tại gần nhất, và một bước **cập nhật (update)**, trong đó mỗi tâm nhảy đến trung bình của các điểm vừa được gán cho nó.

<img src="../animations/kmeans_iterations_demo.gif" alt="Ảnh động k-means: các điểm đổi màu theo tâm gần nhất, rồi các tâm nhảy đến trung bình của các điểm được gán, lặp lại cho đến khi hội tụ" style="display: block; margin: 0 auto; max-width: 680px;">

K-means, còn gọi là thuật toán Lloyd, ở đây giữ nguyên dạng như trong sách giáo khoa: chọn `k` điểm ngẫu nhiên làm tâm ban đầu, gán mọi điểm cho tâm gần nhất, tính lại mỗi tâm bằng trung bình các điểm được gán, lặp lại cho đến khi không còn gì di chuyển. Với các cụm gọn, kích thước gần bằng nhau, gần tròn trong không gian L*a*b* (chính là điều ảnh tổng hợp này có), nó khôi phục các vùng thực gần như hoàn hảo. Có hai lưu ý thực tế đáng nêu ra ngay cả trong trường hợp thành công này: `k` (số cụm) phải được chọn trước — không có gì trong thuật toán tự phát hiện ra nó — và các lần khởi tạo ngẫu nhiên khác nhau có thể hội tụ về các cực trị cục bộ khác nhau, đôi khi tệ hơn thấy rõ (hãy thử đổi `seed=3` thành `seed=0` ở trên và chạy lại); **k-means++** (<a href="../references.html#arthur-2007-kmeanspp">Arthur & Vassilvitskii, 2007</a><span class="landmark-paper">&#9733;</span>) khắc phục điều này bằng cách khởi tạo các tâm sao cho chúng trải xa nhau thay vì hoàn toàn ngẫu nhiên.

## Nơi k-means thất bại: các cụm không lồi (Where k-means breaks: non-convex clusters)
Quy tắc cập nhật của k-means — gán cho *tâm* gần nhất — chỉ có thể chia không gian thành các vùng lồi. Hãy tạo một tập dữ liệu hoàn toàn không lồi: hai vòng tròn đồng tâm. Không có cách đặt hai tâm nào có thể tách chúng đúng, bất kể thuật toán lặp bao nhiêu lần.

In [ ]:
rng = np.random.default_rng(1)

def ring(n, r, noise=0.05):
    theta = rng.uniform(0, 2 * np.pi, n)
    rad = r + rng.normal(0, noise, n)
    return np.stack([rad * np.cos(theta), rad * np.sin(theta)], axis=1)

X_rings = np.vstack([ring(150, 1.0), ring(150, 2.5)])
true_rings = np.array([0] * 150 + [1] * 150)

def best_binary_acc(labels, true):
    return max((labels == true).mean(), (labels == (1 - true)).mean())

km_ring_labels, km_ring_centers = kmeans(X_rings, k=2, seed=0)
print(f'k-means accuracy on concentric rings: {best_binary_acc(km_ring_labels, true_rings):.1%}  (2-class chance = 50%)')

fig, axes = plt.subplots(1, 2, figsize=(7, 3.5))
axes[0].scatter(*X_rings.T, c=true_rings, cmap='coolwarm', s=8)
axes[0].set_title('true rings', fontsize=9); axes[0].set_aspect('equal')
axes[1].scatter(*X_rings.T, c=km_ring_labels, cmap='coolwarm', s=8)
axes[1].scatter(*km_ring_centers.T, c='black', marker='x', s=100, label='centers')
axes[1].set_title('k-means result', fontsize=9); axes[1].set_aspect('equal'); axes[1].legend(fontsize=7)
plt.show()

K-means cho kết quả ngang mức đoán ngẫu nhiên — nó chia hai vòng theo phương xuyên tâm, một nửa mỗi vòng thuộc về mỗi cụm, vì đó là loại ranh giới duy nhất (một đường thẳng, cách đều hai tâm) mà thuật toán có thể vẽ được. Các cụm ở đây hoàn toàn không mơ hồ đối với mắt người; chỉ là *giả định của thuật toán* (các cụm tròn, dựa trên tâm) là công cụ sai cho hình dạng này.

## Mô hình hỗn hợp Gauss: các cụm elip, gán mềm (Gaussian mixture models: soft, elliptical clusters)
**Mô hình hỗn hợp Gauss (Gaussian mixture model — GMM)** tổng quát hóa k-means theo hai cách: mỗi cụm có ma trận hiệp phương sai đầy đủ riêng (nên các cụm có thể là elip thuôn dài, không chỉ là hình tròn), và việc gán là *mềm* — mỗi điểm nhận một xác suất thuộc về mỗi cụm, thay vì một nhãn cứng. Mô hình được khớp bằng thuật toán **kỳ vọng-cực đại hóa (expectation-maximization — EM)** (<a href="../references.html#dempster-1977-em">Dempster, Laird, & Rubin, 1977</a><span class="landmark-paper">&#9733;</span>): bước E tính trách nhiệm (responsibility — xác suất) của mỗi điểm theo các hàm Gauss hiện tại; bước M khớp lại trung bình, hiệp phương sai và trọng số của mỗi hàm Gauss theo các điểm đã được đánh trọng số trách nhiệm.

In [ ]:
def gmm_em(X, k, n_iter=100, seed=0):
    rng = np.random.default_rng(seed)
    n, d = X.shape
    means = X[rng.choice(n, k, replace=False)].copy()
    covs = np.array([np.cov(X.T) + 1e-3 * np.eye(d) for _ in range(k)])
    weights = np.full(k, 1 / k)
    for _ in range(n_iter):
        resp = np.zeros((n, k))
        for j in range(k):
            diff = X - means[j]
            inv = np.linalg.inv(covs[j])
            expo = -0.5 * np.sum(diff @ inv * diff, axis=1)
            norm = 1.0 / np.sqrt((2 * np.pi) ** d * np.linalg.det(covs[j]))
            resp[:, j] = weights[j] * norm * np.exp(expo)
        resp /= resp.sum(1, keepdims=True) + 1e-12
        Nk = resp.sum(0)
        weights = Nk / n
        means = (resp.T @ X) / Nk[:, None]
        for j in range(k):
            diff = X - means[j]
            covs[j] = (resp[:, j:j + 1] * diff).T @ diff / Nk[j] + 1e-6 * np.eye(d)
    return resp.argmax(1), means, covs

gmm_ring_labels, gmm_means, gmm_covs = gmm_em(X_rings, k=2, seed=0)
print(f'GMM accuracy on concentric rings: {best_binary_acc(gmm_ring_labels, true_rings):.1%}')

GMM ở đây không làm tốt hơn k-means — cũng gần mức đoán ngẫu nhiên. Đó là một điểm quan trọng, dễ bỏ sót: GMM khắc phục giả định *chỉ-tròn* của k-means, nhưng nó vẫn là mô hình **đơn đỉnh trên mỗi cụm (unimodal-per-cluster)** — mỗi cụm được mô tả bằng một ụ Gauss duy nhất. Một vòng tròn không phải hình elip cũng như không phải hình tròn đặc; không có hàm Gauss đơn lẻ nào, dù hình dạng nào, khớp tốt với một hình vành khuyên. GMM và k-means thất bại trên tập dữ liệu này vì cùng một lý do gốc rễ.

## DBSCAN: cụm được xác định bởi mật độ, không phải hình dạng (DBSCAN: clusters defined by density, not shape)
**DBSCAN** (phân cụm không gian dựa trên mật độ, <a href="../references.html#ester-1996-dbscan">Ester et al., 1996</a><span class="landmark-paper">&#9733;</span>) hoàn toàn không khớp một hình dạng tham số nào. Một điểm là **điểm lõi (core point)** nếu có ít nhất `min_samples` điểm khác nằm trong khoảng cách `eps` từ nó. Các cụm được hình thành bằng cách nối chuỗi các điểm lõi nằm trong phạm vi `eps` của nhau (cùng các lân cận của chúng), nên một cụm có thể có bất kỳ hình dạng nào — kể cả hình vòng — miễn là nó là một vùng *liên thông, đủ dày đặc* của không gian. Các điểm không nằm trong vùng lân cận của điểm lõi nào được gán nhãn là **nhiễu (noise)**, thay vì bị ép vào cụm gần nhất.

Cơ chế phát triển cụm ở đây cùng ý tưởng với tô loang ở Bài 4: cả hai đều duy trì một biên gồm các điểm ảnh/điểm còn cần thăm, và chỉ mở rộng nó tới các lân cận có thể đi tới ngay lập tức cho đến khi không còn gì mới có thể đi tới. Hai cách chỉ khác nhau ở ý nghĩa của "có thể đi tới": tô loang mở rộng tới các lân cận lưới liên thông 4/8, còn DBSCAN mở rộng tới bất kỳ điểm nào nằm trong phạm vi `eps` của một điểm lõi.

In [ ]:
def dbscan(X, eps, min_samples):
    n = len(X)
    labels = np.full(n, -1)
    visited = np.zeros(n, dtype=bool)
    dist = np.sqrt(((X[:, None, :] - X[None, :, :]) ** 2).sum(-1))
    cluster_id = 0
    for i in range(n):
        if visited[i]:
            continue
        visited[i] = True
        neighbors = list(np.where(dist[i] <= eps)[0])
        if len(neighbors) < min_samples:
            continue  # stays noise (-1)
        labels[i] = cluster_id
        j = 0
        while j < len(neighbors):
            q = neighbors[j]
            if not visited[q]:
                visited[q] = True
                q_neighbors = np.where(dist[q] <= eps)[0]
                if len(q_neighbors) >= min_samples:
                    neighbors.extend([x for x in q_neighbors if x not in neighbors])
            if labels[q] == -1:
                labels[q] = cluster_id
            j += 1
        cluster_id += 1
    return labels

db_ring_labels = dbscan(X_rings, eps=0.6, min_samples=4)
n_clusters = len(set(db_ring_labels.tolist()) - {-1})
n_noise = (db_ring_labels == -1).sum()
print(f'DBSCAN found {n_clusters} clusters, {n_noise} noise points')
print(f'DBSCAN accuracy on concentric rings: {best_binary_acc(db_ring_labels, true_rings):.1%}')

fig, axes = plt.subplots(1, 3, figsize=(10, 3.5))
for ax, labels, title in zip(axes, [true_rings, km_ring_labels, db_ring_labels],
                              ['true rings', 'k-means (fails)', 'DBSCAN (succeeds)']):
    ax.scatter(*X_rings.T, c=labels, cmap='coolwarm', s=8)
    ax.set_title(title, fontsize=9); ax.set_aspect('equal')
plt.show()

Ảnh động này cho thấy quá trình phát triển vùng của DBSCAN trên cùng tập dữ liệu hai vòng: một biên được tô sáng (viền cam), mỗi điểm có vùng lân cận bán kính `eps` được vẽ thành một vòng tròn xám mờ, lan ra từ mỗi điểm lõi, thăm và chiếm lấy các lân cận liên thông mật độ theo từng đợt, trong khi các điểm nằm ngoài phạm vi `eps` của mọi điểm lõi được để màu xám và cuối cùng bị đánh dấu là nhiễu.

<img src="../animations/dbscan_growth_demo.gif" alt="Ảnh động phát triển vùng DBSCAN trên tập dữ liệu hai vòng đồng tâm, với biên mở rộng dần và vùng lân cận bán kính eps chiếm lấy các điểm liên thông mật độ theo từng đợt" style="display: block; margin: 0 auto; max-width: 680px;">

DBSCAN tách hai vòng một cách hoàn hảo — nó không bao giờ giả định một tâm hay một hình dạng, chỉ giả định rằng các điểm *bên trong* một vòng liên thông dày đặc với các lân cận của chúng, còn khoảng trống giữa các vòng thì không. Điều này đi kèm cái giá thực sự: `eps` và `min_samples` phải được chọn thủ công (lựa chọn kém sẽ chia mỗi vòng thành hàng chục cung nhỏ — hãy thử `eps=0.15` ở trên để thấy điều đó xảy ra), và DBSCAN gặp khó khi các cụm thực có mật độ rất khác nhau, vì một `eps` toàn cục không thể đồng thời phù hợp cho cả cụm thưa lẫn cụm dày.

## Khả năng chống nhiễu: một lợi thế thực sự của việc "không ép gán" (Noise robustness: a real advantage of "no forced assignment")
Thêm các điểm lộn xộn — các điểm rải đều trên toàn vùng, không thuộc vòng nào — và so sánh cách mỗi thuật toán xử lý chúng. k-means và GMM không có khái niệm "không thuộc cụm nào": mọi điểm đều bị gán vào đâu đó, kéo các tâm cụm về phía các điểm lộn xộn. DBSCAN có thể gọi đúng tên điểm lộn xộn.

In [ ]:
noise_pts = rng.uniform(-3, 3, (60, 2))
X_cluttered = np.vstack([X_rings, noise_pts])
true_cluttered = np.concatenate([true_rings, np.full(60, -1)])  # -1 = "not really a ring"

km_clut_labels, _ = kmeans(X_cluttered, k=2, seed=0)
db_clut_labels = dbscan(X_cluttered, eps=0.6, min_samples=4)

km_noise_correctly_flagged = 0  # k-means has no noise concept at all
db_noise_correctly_flagged = (db_clut_labels[300:] == -1).sum()

print(f'{"":>10} {"noise points correctly flagged":>32} {"out of":>8}')
print(f'{"k-means":>10} {km_noise_correctly_flagged:>32} {60:>8}  (no noise concept — every point forced into a cluster)')
print(f'{"DBSCAN":>10} {db_noise_correctly_flagged:>32} {60:>8}')

Theo cách xây dựng, k-means đánh dấu đúng 0 trên 60 điểm lộn xộn là bất cứ thứ gì khác ngoài "chắc chắn thuộc một vòng" — thuật toán không có từ vựng cho "không thuộc về đâu cả". DBSCAN cô lập đúng một phần thiểu số trong số đó; phần còn lại tình cờ rơi vào phạm vi `eps` của một điểm vòng thật (hoặc của nhau, tạo thành một cụm nhỏ riêng), đó là một kết quả hợp lý, không phải lỗi — phân cụm dựa trên mật độ chỉ gọi một điểm là nhiễu nếu nó thực sự bị cô lập, và các điểm rải ngẫu nhiên đều vẫn thỉnh thoảng tạo ra các cụm nhỏ. Điểm định tính vẫn đúng bất kể con số chính xác: DBSCAN có một hạng mục thực sự, dùng được cho "thứ này không khớp vào đâu cả", trong khi k-means về mặt cấu trúc thì không thể.

## Tiếp theo là gì (Where this goes next)
Đây là phần kết thúc của Phần 1. Nhiều thuật toán gặp từ đầu đến giờ hoạt động trực tiếp trên **các đặc trưng thô hoặc được chọn thủ công** — màu điểm ảnh, tọa độ `(x, y)`, kết cấu — và dựa vào các quy tắc cụ thể, chẳng hạn như các hàm khoảng cách do con người chọn được dùng ở đây. Các bài sau sẽ quay lại nhiều bài toán trong số này, dùng học sâu (deep learning) để tự động khám phá các đặc trưng và quy tắc kết hợp chúng. Ví dụ, phân đoạn thực thể (instance segmentation) ở Bài 43 dùng một ý tưởng phân cụm tương tự, và bộ tự mã hóa (autoencoder) ở Bài 48 tự học chính không gian đặc trưng, hoàn toàn từ sai số tái tạo, trước khi có bất kỳ yêu cầu phân nhóm nào. Trong khi các phương pháp của bài này giả định một hình dạng rồi khớp nó, các phương pháp học sâu ở phần sau của khóa học học một biểu diễn và để cấu trúc tự nổi lên từ đó.

### Bài tập (Exercises)
1. Chạy lại k-means phân đoạn màu với `k=3` và `k=6` thay vì giá trị đúng `4`. Vì không có gì trong thuật toán có thể "biết" số cụm đúng, phân đoạn suy giảm thế nào theo mỗi chiều, và bạn có thể chỉ nhìn đầu ra mà biết `k` nào có khả năng sai không?
2. GMM ở trên chỉ được đánh giá trên tập dữ liệu hai vòng, nơi nó thất bại vì cùng lý do với k-means. Hãy dựng một tập dữ liệu hai cụm trong đó các cụm thực sự là hình elip (ví dụ `np.random.multivariate_normal` với ma trận hiệp phương sai không chéo) và so sánh GMM với k-means ở đó. Lần này, sự linh hoạt bổ sung của GMM (khớp hình dạng, không chỉ tâm) có cho thấy lợi thế độ chính xác thực sự không?
3. Trong ví dụ DBSCAN với điểm lộn xộn, thu nhỏ `eps` từ `0.6` xuống `0.3`. DBSCAN có bắt đầu gán nhầm các điểm vòng thật là nhiễu, các điểm lộn xộn thật là thuộc một vòng, hay cả hai — và điều đó có khớp với cảnh báo trước đó rằng `eps` là một lựa chọn tinh chỉnh thủ công, đặc thù cho tập dữ liệu, chứ không phải thứ thuật toán tự khám phá không?